
# 📈 Elasticity Modeling & Extraction (GLM + Shrinkage)

**Objective:** Train the price elasticity model using a Generalized Linear Model (Negative Binomial) and extract coefficients to calculate the final price elasticity for each SKU/Context.

**Key Methodologies:**
1. **Categorical Transformation:** K-Means clustering for price tiers and Pareto principle for cardinality reduction.
2. **Modeling:** GLM (Negative Binomial) with ElasticNet Regularization (Step 1) and Unpenalized fit (Step 2) for inference.
3. **Diagnostics:** Comprehensive residual analysis, Partial Dependence Plots (PDP), and VIF checks.
4. **Post-Processing (Shrinkage):** Application of **Bühlmann-Straub Credibility** to adjust local elasticities towards the group mean based on variance and sample size.

**Output:** `pricing_db.gold_elasticity_model_trusted`


## 📚 1. Libraries & Setup

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy
import shap
import os
import re
import math
import gc
import csv
from scipy import stats
from scipy.stats import norm, probplot, chi2, linregress, zscore
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from statsmodels.genmod.generalized_linear_model import GLMResults
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.nonparametric.smoothers_lowess import lowess
from typing import Tuple, List

# PySpark
from pyspark.sql import functions as F, Window, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.ml.feature import VectorAssembler, RFormula
from pyspark.ml.clustering import KMeans

# Configuration
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.precision', 2)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")


## 📥 2. Data Preparation & Advanced Feature Engineering

In [ ]:
# Load Data
df_raw = spark.table("pricing_db.silver_feature_store_refined")

In [ ]:
# Configuration for Categorical Analysis 
# (Anonymized dictionary - simulating the output of the Feature Selection step)
categorical_analysis_dict = {
    'REGION_STATE': {
        'total_original': 12,
        'n_kept': 11,
        'percent_kept': 91.66,
        'levels_kept': ['STATE_A', 'STATE_B', 'STATE_C', 'STATE_D', 'STATE_E', 'STATE_F', 'STATE_G'], # Anonymized
        'levels_removed': ['others']
    },
    'BUSINESS_UNIT': {
        'total_original': 4,
        'n_kept': 4,
        'percent_kept': 100.0,
        'levels_kept': ['BU_1', 'BU_2', 'BU_3', 'BU_4'],
        'levels_removed': []
    },
    'BRAND': {
        'total_original': 43,
        'n_kept': 42,
        'percent_kept': 97.67,
        'levels_kept': ['BRAND_A', 'BRAND_B', 'BRAND_C', 'BRAND_D'], # ... truncated list
        'levels_removed': ['others']
    },
    # ... other variables
}

In [ ]:
def process_categorical_transformations(
    df: DataFrame,
    cat_cols: list,
    analysis_dict: dict = None,
    price_col: str = 'UNIT_PRICE',
    brand_col: str = 'BRAND',
    volume_col: str = 'SALES_VOLUME',
    pareto_threshold: float = 0.80,
    pareto_threshold_brand: float = 0.95,
    n_tiers: int = None,
    apply_tiers: bool = True,
    apply_pareto: bool = True,
    apply_whitelist: bool = True) -> DataFrame:
    """
    Orchestrates advanced categorical transformations:
    1. **Price Tiers (K-Means):** Clusters brands into price tiers. Uses Elbow Method if n_tiers is None.
    2. **Pareto (Cardinality Reduction):** Groups tail categories into 'Others' based on cumulative volume.
    3. **Whitelist Filtering:** Aligns categories with the Feature Selection step.

    Returns:
        DataFrame with new columns suffixed with '_proc' (original columns preserved).
    """
    
    df_proc = df

    # Helper to define processed column name
    def get_proc_col_name(col_name):
        return f"{col_name}_reduced" if col_name == "BRAND_CATEGORY" else f"{col_name}_proc"

    # ==============================================================================
    # 1. Price Tiers Generation (Unsupervised Learning: K-Means)
    # ==============================================================================
    if apply_tiers and (brand_col in df_proc.columns):
        # Aggregate average price per brand
        df_price_avg = df_proc.groupBy(brand_col).agg(F.mean(price_col).alias("avg_price")).dropna()
        assembler = VectorAssembler(inputCols=["avg_price"], outputCol="features")
        df_kmeans_input = assembler.transform(df_price_avg).cache()
        
        k_final = n_tiers
        
        # Elbow Method Logic to find optimal K if not provided
        if k_final is None:
            costs = {}
            k_min, k_max = 2, 7
            n_brands = df_kmeans_input.count()
            k_max = min(k_max, n_brands - 1)
            
            if k_max < k_min:
                k_final = 2
            else:
                for k in range(k_min, k_max + 1):
                    kmeans_temp = KMeans(k=k, seed=42, featuresCol="features")
                    model_temp = kmeans_temp.fit(df_kmeans_input)
                    costs[k] = model_temp.summary.trainingCost

                # Geometric calculation of the "Elbow" point
                p1 = (k_min, costs[k_min])
                p2 = (k_max, costs[k_max])
                best_k = k_min
                max_dist = -1
                
                for k, cost in costs.items():
                    numerator = abs((p2[1] - p1[1]) * k - (p2[0] - p1[0]) * cost + p2[0] * p1[1] - p2[1] * p1[0])
                    denominator = math.sqrt((p2[1] - p1[1])**2 + (p2[0] - p1[0])**2)
                    dist = numerator / denominator
                    if dist > max_dist:
                        max_dist = dist
                        best_k = k
                k_final = best_k
                print(f"-> Elbow Method determined optimal K = {k_final} for {brand_col}")

        # Final K-Means Training
        kmeans_final = KMeans(k=k_final, seed=42, featuresCol="features", predictionCol="cluster_id")
        model = kmeans_final.fit(df_kmeans_input)
        
        # Mapping clusters to Tiers (0 = Lowest Price)
        centers = model.clusterCenters()
        sorted_centers = sorted([(i, c[0]) for i, c in enumerate(centers)], key=lambda x: x[1])
        
        mapping_dict = {old_id: f"TIER_{new_rank}" for new_rank, (old_id, _) in enumerate(sorted_centers)}
        mapping_expr = F.create_map([F.lit(x) for x in sum(mapping_dict.items(), ())])
        
        df_predictions = model.transform(df_kmeans_input)
        df_tiers = df_predictions.withColumn(
            f"TIER_{brand_col}", 
            mapping_expr.getItem(F.col("cluster_id"))).select(brand_col, f"TIER_{brand_col}")
        
        df_proc = df_proc.join(df_tiers, on=brand_col, how="left")
        df_kmeans_input.unpersist()

    # ==============================================================================
    # 2. Pareto Grouping (80/20 Rule)
    # ==============================================================================
    if apply_pareto and cat_cols:
        total_vol = df_proc.select(F.sum(volume_col)).first()[0]
        
        for col_name in cat_cols:
            threshold = pareto_threshold_brand if col_name == brand_col else pareto_threshold

            df_vol = df_proc.groupBy(col_name).agg(F.sum(volume_col).alias("vol_cat"))
            w = Window.orderBy(F.desc("vol_cat"))
            df_cum = df_vol.withColumn("cum_vol", F.sum("vol_cat").over(w))
            
            # Identify categories to KEEP
            df_keep = df_cum.filter(F.col("cum_vol") <= (total_vol * threshold)) \
                            .select(F.col(col_name).alias(f"{col_name}_key"))
            
            # Broadcast join for efficiency
            df_proc = df_proc.join(F.broadcast(df_keep), 
                                   df_proc[col_name] == df_keep[f"{col_name}_key"], 
                                   "left")
            
            target_col = get_proc_col_name(col_name)
            
            df_proc = df_proc.withColumn(
                target_col,
                F.when(F.col(f"{col_name}_key").isNotNull(), F.col(col_name))
                .otherwise(F.lit("Others"))).drop(f"{col_name}_key")

    # ==============================================================================
    # 3. Whitelist Enforcement (Feature Selection Alignment)
    # ==============================================================================
    if apply_whitelist and analysis_dict:
        for var_name, analysis in analysis_dict.items():
            target_col = get_proc_col_name(var_name)
            source_col = target_col if target_col in df_proc.columns else var_name
            
            if source_col in df_proc.columns:
                levels_kept = analysis.get('levels_kept')
                if levels_kept:
                    df_proc = df_proc.withColumn(
                        target_col, 
                        F.when(F.col(source_col).isin(levels_kept), F.col(source_col))
                        .otherwise(F.lit("Others")))

    return df_proc

In [ ]:
mlflow.autolog(disable=True)

categorical_cols = ['BRAND', 'SUB_BUSINESS_UNIT', 'REGION_STATE', 'SUB_CHANNEL', 
    'SALES_TEAM', 'month', 'EAN_CODE']

# Apply transformations
df_processed = process_categorical_transformations(
    df=df_raw,
    cat_cols=categorical_cols,
    analysis_dict=categorical_analysis_dict,
    price_col='NET_PRICE_UNIT',
    brand_col='BRAND',
    volume_col='SALES_VOLUME',
    n_tiers=50, # Manual override for granularity
    apply_tiers=True,
    apply_pareto=True,
    apply_whitelist=True)

In [ ]:
# Validation of transformations
results = []
for col in categorical_cols:
    col_proc = f"{col}_proc"
    count_orig = df_processed.select(col).distinct().count()
    count_proc = df_processed.select(col_proc).distinct().count()
    results.append((col, count_orig, col_proc, count_proc))

display(spark.createDataFrame(results, ["original_col", "n_original", "processed_col", "n_processed"]))

In [ ]:
def prepare_modeling_df(df: DataFrame, volume_col: str = 'Q26', cat_vars: list = None, log_Q = False) -> DataFrame:
    """
    Final feature engineering: 
    1. Defines target Q.
    2. Log-transforms price.
    3. Creates Cyclical Seasonality (Sin/Cos).
    4. Casts categoricals to string.
    """
    if log_Q:
        df2 = df.withColumn('Q', F.log(F.col(volume_col))) \
                .withColumn('ln_price', F.log(F.col('NET_PRICE_UNIT'))) \
                .withColumn(
                    'sin_w',
                    F.sin(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52)) \
                .withColumn(
                    'cos_w',
                    F.cos(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52))
    else:
        df2 = df.withColumn('Q', F.col(volume_col)) \
                .withColumn('ln_price', F.log(F.col('NET_PRICE_UNIT'))) \
                .withColumn(
                    'sin_w',
                    F.sin(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52)) \
                .withColumn(
                    'cos_w',
                    F.cos(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52))

    # Cast variables
    for col in cat_vars:
        df2 = df2.withColumn(col, F.col(col).cast('string'))

    # Select final columns
    cols_final = ['Q', 'ln_price', 'sin_w', 'cos_w', 'MARKET_PRICE_GAP', 'MARKET_PRICE'] + cat_vars + ['CAMPAIGN_ID', 'ORDER_ID']
    model_df = df2.select(*cols_final)
    
    return model_df

In [ ]:
# Define variables for the model
cat_vars = ['EAN_CODE_proc', 'EAN_CODE', 
    'BRAND_proc', 'TIER_BRAND',
    'SUB_BUSINESS_UNIT_proc', 'REGION_STATE_proc', 'SUB_CHANNEL_proc',
    'BRAND', 'REGION_STATE', 'BUSINESS_UNIT', 'SUB_BUSINESS_UNIT', 
    'BRAND_CATEGORY', 'SUB_CHANNEL', 
    'month', 'month_proc',
    'day_of_week', 'price_change', 'week_of_year',
    'BRAND_CATEGORY_reduced']

# Prepare final dataframe
model_df = prepare_modeling_df(df_processed, 
                             volume_col='SALES_VOLUME', # Using raw sales volume instead of Q26 based on testing
                             cat_vars=cat_vars, 
                             log_Q=False) 
model_df = model_df.withColumn('week_of_year', F.col('week_of_year').cast('int'))


## 📊 3. Model Training & Diagnostics

In [ ]:
# MLflow Setup
experiment_name = "/Shared/smart_pricing_elasticity_model_prod"
mlflow.set_experiment(experiment_name)

In [ ]:
def build_centered_design_matrix(
    df: DataFrame,
    categorical_vars: list,
    seasonal_vars: list,
    custom_interactions: list,
    include_categorical_main_effects: bool = True,
    include_seasonal_main_effects: bool = True,
    volume_col: str = "Q",
    price_col: str = "ln_price",
    price_interaction_exclusion_list: list[str] = ['MARKET_PRICE_GAP']) -> Tuple[str, DataFrame, float, str]:
    """
    Constructs the design matrix with **centered price** to mitigate structural multicollinearity.
    
    Returns:
        formula_str: R-style formula.
        df_feat: Spark DataFrame with 'features' vector.
        price_mean: Mean value used for centering.
        centered_price_col: Name of the centered column.
    """
    # 1) Center the Price Variable
    price_mean = df.select(F.mean(price_col)).first()[0]
    centered_price_col = f"{price_col}_c"
    df = df.withColumn(centered_price_col, F.col(price_col) - F.lit(price_mean))

    # 2) Convert Categoricals to String
    for c in categorical_vars:
        df = df.withColumn(c, F.col(c).cast("string"))

    # 3) Clean Custom Interactions (Regex)
    clean_custom = [
        re.sub(r"C\(([^)]+)\)", r"\1", term)
        for term in custom_interactions]

    # 4) Assemble Formula Terms
    main_effects = [centered_price_col]
    if include_seasonal_main_effects:
        main_effects += seasonal_vars
    if include_categorical_main_effects:
        main_effects += categorical_vars

    # Interactions with Price
    interactions = [
        f"{centered_price_col}:{v}" for v in categorical_vars
        if v not in price_interaction_exclusion_list] + [
        f"{centered_price_col}:{s}" for s in seasonal_vars
        if s not in price_interaction_exclusion_list]

    # 5) Join Terms
    all_terms = main_effects + interactions + clean_custom
    formula_str = f"{volume_col} ~ " + " + ".join(all_terms)

    # 6) Generate Features Vector via RFormula
    rf = RFormula(
        formula=formula_str,
        featuresCol="features",
        labelCol=volume_col)
    df_feat = rf.fit(df).transform(df)

    return formula_str, df_feat, price_mean, centered_price_col

In [ ]:
# Variable Selection for Final Model
final_categorical_vars = ['TIER_BRAND',
    'REGION_STATE_proc',
    'SUB_CHANNEL_proc', 
    'SALES_TEAM_proc', 
    'month_proc',]
final_seasonal_vars = []
custom_interactions = []
price_col = 'ln_price'

# Sampling for Modeling (Training on full dataset might OOM on driver)
total_count = model_df.filter(F.col('Q') >= 1).count()
fraction = min(4000000 / total_count, 1.0) # Target 4M rows
filtered_df = model_df.filter(F.col('Q') >= 1).sample(withReplacement=False, fraction=fraction, seed=42)

# Build Matrix
formula, df_feat, price_mean, centered_price_col_name = build_centered_design_matrix(
     df=filtered_df,
     categorical_vars=final_categorical_vars,
     seasonal_vars=final_seasonal_vars,
     custom_interactions=custom_interactions,
     include_categorical_main_effects=True,
     include_seasonal_main_effects=True,
     volume_col='Q',
     price_col=price_col)

# Convert to Pandas for Statsmodels
pandas_df = df_feat.toPandas()

print(f"--- Formula ---\n{formula}\n")
print(f"Dataset Shape: {pandas_df.shape}")


### 🔍 Variance Analysis (GLM Family Selection)

In [ ]:
q_df = df_feat.select("Q").toPandas().dropna()

# Binning data to estimate Mean-Variance relationship
stats_agg = (
    q_df.groupby(pd.qcut(q_df['Q'], q=20, duplicates='drop'))['Q']
    .agg(['mean', 'var'])
    .query('var > 0 and mean > 0'))

log_mean = np.log(stats_agg['mean'])
log_var = np.log(stats_agg['var'])

slope, _, _, _, _ = linregress(log_mean, log_var)

print("-" * 70)
print(f"Variance Power Estimate (p): {slope:.4f}")
print("-" * 70)

if 0.95 < slope < 1.05:
    rec = "Poisson (p ≈ 1)"
elif 1.05 <= slope < 1.95:
    rec = "Negative Binomial or Tweedie (1 < p < 2)"
elif 1.95 <= slope < 2.2:
    rec = "Gamma or Negative Binomial (p ≈ 2) - Ideal for our overdispersed data"
else:
    rec = "Tweedie with p > 2"

print(f"Recommendation: {rec}")

In [ ]:
def train_glm_model(regularize: bool, best_params: dict):
    """
    Trains the GLM model (Negative Binomial), logs metrics to MLflow, 
    and performs extensive statistical diagnostics (12 Steps).
    """

    # --- Start MLflow Run ---
    logged_dataframes = {}

    with mlflow.start_run():
        print("✅ Starting MLflow Run...")
        print("--- Design Matrix Formula ---")
        print(formula)
        print("-" * 30)

        # =========================================================================================
        # STEP 1: LOGGING PARAMETERS
        # =========================================================================================
        print("📝 1/12: Logging model parameters...")
        mlflow.log_param("regularized", regularize)
        mlflow.log_param("model_type", "NegativeBinomial (log link)")
        mlflow.log_param("formula", formula)
        # Assuming categorical_vars, seasonal_vars, etc. are defined in global scope or passed
        mlflow.log_param("categorical_vars", str(final_categorical_vars)) 
        mlflow.log_param("seasonal_vars", str(final_seasonal_vars))
        mlflow.log_param("custom_interactions", str(custom_interactions))
        mlflow.log_param("sample_size", len(pandas_df))
        
        # --- Price Centering Params ---
        mlflow.log_param("original_price_col", price_col)
        mlflow.log_param("centered_price_col", centered_price_col_name)
        mlflow.log_param("price_mean_for_centering", price_mean)
        mlflow.log_param("model_family", 'NegativeBinomial')

        # Save formula as text artifact
        with open("formula.txt", "w") as f:
            f.write(formula)
        mlflow.log_artifact("formula.txt", artifact_path="diagnostics")


        # =========================================================================================
        # STEP 2: MODEL TRAINING
        # =========================================================================================
        fam_name = 'Negative Binomial'

        # Define GLM with Negative Binomial family and explicit log link
        nb_family = sm.families.NegativeBinomial(link=sm.families.links.log())

        if regularize:
            print(f"🚀 2/12: Training Regularized {fam_name} GLM (ElasticNet)...")
            
            glm_nb = smf.glm(
                formula=formula,
                data=pandas_df,
                family=nb_family)        

            print("   -> Step 2.1: Regularized fit to stabilize coefficients...")
            results_reg = glm_nb.fit_regularized(
                method='elastic_net', 
                alpha=best_params['alpha'],
                L1_wt=best_params['L1_wt'], # Mixing param (L1 vs L2)
                refit=False # We refit manually below for full stats)
            
            print("   -> Step 2.2: Final Unpenalized fit for inference (using started params)...")
            # Extract starting params and delete regularization object to free memory
            start_params_opt = results_reg.params
            del results_reg # [MEMORY OPTIMIZATION]
            gc.collect()    # [MEMORY OPTIMIZATION]

            results = glm_nb.fit(start_params=start_params_opt, cov_type='HC1')

        else:
            print(f"🚀 2/12: Training Unpenalized {fam_name} GLM...")
            glm_nb = smf.glm(
                formula=formula,
                data=pandas_df,
                family=nb_family)
                
            results = glm_nb.fit(cov_type='HC1')
            print("   -> Training complete.")
        
        # Post-training cleanup
        gc.collect() 


        # =========================================================================================
        # STEP 3: PREDICTIONS & ERROR METRICS (SCIKIT-LEARN)
        # =========================================================================================
        print("📉 3/12: Calculating predictions and error metrics...")

        y_true = pandas_df['Q']
        y_pred = results.predict(pandas_df)

        # --- Metrics Calculation ---
        
        # RMSE
        rmse = mean_squared_error(y_true, y_pred, squared=False)

        # MAE
        mae = mean_absolute_error(y_true, y_pred)

        # MAPE
        mape = mean_absolute_percentage_error(y_true, y_pred)

        # sMAPE (Symmetric MAPE)
        error = y_pred - y_true 
        denominator_smape = np.abs(y_true) + np.abs(y_pred) + 1e-8
        smape = np.mean(2 * np.abs(error) / denominator_smape)

        # WMAPE (Weighted MAPE)
        wmape = np.sum(np.abs(error)) / np.sum(np.abs(y_true))
        
        # [MEMORY OPTIMIZATION]
        del error, denominator_smape
        gc.collect()

        print("   -> Metrics calculated.")

        # Logging
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("MAPE_pct", mape * 100)
        mlflow.log_metric("sMAPE_pct", smape * 100)
        mlflow.log_metric("WMAPE_pct", wmape * 100)


        # =========================================================================================
        # STEP 4: GOODNESS OF FIT METRICS (STATSMODELS)
        # =========================================================================================
        print("📈 4/12: Calculating Goodness of Fit (AIC, BIC, R2)...")
        
        log_likelihood = results.llf
        aic = results.aic
        bic = results.bic
        deviance = results.deviance
        null_deviance = results.null_deviance
        dispersion = results.scale

        # McFadden's Pseudo R-Squared
        pseudo_r2 = 1 - (results.llf / results.llnull)

        # Logging
        mlflow.log_metric("log_likelihood", log_likelihood)
        mlflow.log_metric("Pseudo_R2_McFadden", pseudo_r2)
        mlflow.log_metric("AIC", aic)
        mlflow.log_metric("BIC", bic)
        mlflow.log_metric("Deviance", deviance)
        mlflow.log_metric("Dispersion_Scale", dispersion)

        print(f"     -> Pseudo R2: {pseudo_r2:.4f}")
        print(f"     -> Information Criteria: AIC={aic:.2f}, BIC={bic:.2f}")

        if pseudo_r2 > 0.02:
            print(f"     -> Verdict: Model shows explanatory power (Pseudo R2 > 0.02).")
        else:
            print(f"     -> WARNING: Pseudo R2 is very low. Model may lack explanatory power.")


        # =========================================================================================
        # STEP 5: COEFFICIENT ANALYSIS
        # =========================================================================================
        print("📋 5/12: Generating Coefficients DataFrame...")

        # 1. Extract results
        df_coef = pd.DataFrame({
            "estimate": results.params,
            "std_err": results.bse,
            "z_score": results.tvalues,
            "p_value": results.pvalues
        })

        # 2. Add Confidence Intervals (95%)
        conf_int = results.conf_int()
        conf_int.columns = ['conf_int_lower', 'conf_int_upper']
        df_coef = df_coef.join(conf_int)

        # 3. Significance Flag (p < 0.05)
        alpha_sig = 0.05
        df_coef['significant'] = df_coef['p_value'] < alpha_sig

        # 4. Reset index to 'term' column
        df_coef.reset_index(inplace=True)
        df_coef.rename(columns={'index': 'term'}, inplace=True)

        print("     -> Coefficients DataFrame created.")


        # =========================================================================================
        # STEP 6: SUMMARY ARTIFACTS & BASE ELASTICITY
        # =========================================================================================
        print("📋 6/12: Extracting Base Elasticity and logging artifacts...")

        try:
            # Extract Base Elasticity (Coefficient of centered price)
            base_elasticity = df_coef.loc[df_coef['term'] == centered_price_col_name, 'estimate'].iloc[0]
            
            mlflow.log_metric(f"Base_Elasticity_{centered_price_col_name}", base_elasticity)
            print(f"     -> Base Price Elasticity (Beta of {centered_price_col_name}): {base_elasticity:.4f}")

        except IndexError:
            print(f"     -> WARNING: Could not find coefficient for '{centered_price_col_name}'. Metric not logged.")

        # Save Coefficients CSV
        csv_path = "model_summary_extended.csv"
        df_coef.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path, artifact_path="diagnostics")
        print(f"     -> Coefficients table saved as artifact.")

        logged_dataframes['coefficients_summary'] = df_coef


        # =========================================================================================
        # STEP 7: FEATURE IMPORTANCE (Z-SCORE)
        # =========================================================================================
        print("⭐ 7/12: Analyzing Feature Importance...")

        # --- Part 1: Individual Importance ---
        df_importance = df_coef[['term', 'z_score']].copy()
        df_importance['importance'] = np.abs(df_importance['z_score'])
        df_importance = df_importance[df_importance['term'] != 'Intercept']
        df_importance_sorted = df_importance.sort_values(by='importance', ascending=False)
        
        logged_dataframes['feature_importance'] = df_importance_sorted
        
        # Plot Top 20
        top_n = 20
        df_plot = df_importance_sorted.head(top_n)
        fig_imp, ax = plt.subplots(figsize=(10, 8))
        sns.barplot(x='importance', y='term', data=df_plot, palette='viridis', ax=ax)
        ax.set_title(f'Top {top_n} Features by Importance (|z-score|)')
        ax.set_xlabel('Importance (|z-score|)')
        plt.tight_layout()
        mlflow.log_figure(fig_imp, "visuals/04_feature_importance_zscore.png")
        plt.close(fig_imp)
        
        del df_plot; gc.collect() # [MEMORY OPTIMIZATION]

        # --- Part 2: Distribution by Group (Boxplot) ---
        print("    -> Generating importance distribution boxplots...")

        def map_term_to_group(term):
            # Removes specific levels like [T.BrandA] to get the variable name
            return re.sub(r'\[T\..*?\]', '', term)

        df_dist_imp = df_importance.copy()
        df_dist_imp['feature_group'] = df_dist_imp['term'].apply(map_term_to_group)
        
        # Filter groups with > 1 member
        group_counts = df_dist_imp['feature_group'].value_counts()
        valid_groups = group_counts[group_counts > 1].index.tolist()
        df_boxplot = df_dist_imp[df_dist_imp['feature_group'].isin(valid_groups)]

        if not df_boxplot.empty:
            median_order = df_boxplot.groupby('feature_group')['importance'].median().sort_values(ascending=False).index
            
            # Dynamic height
            fig_h = max(6, len(median_order) * 0.5)
            
            fig_box, ax = plt.subplots(figsize=(12, fig_h))
            sns.boxplot(x='importance', y='feature_group', data=df_boxplot, order=median_order, palette='viridis', ax=ax)
            ax.set_title('Feature Importance Distribution by Group')
            ax.set_xlabel('Importance (|z-score|)')
            plt.tight_layout()
            
            mlflow.log_figure(fig_box, "visuals/05_importance_distribution_boxplot.png")
            plt.close(fig_box)
        
        del df_dist_imp, df_boxplot; gc.collect()


        # =========================================================================================
        # STEP 8: RESIDUAL DIAGNOSTICS
        # =========================================================================================
        print("🔍 8/12: Generating Residual Diagnostics...")

        # --- 1. Residuals DataFrame ---
        df_resid = pd.DataFrame({
            "Q": pandas_df['Q'],
            "prediction": results.fittedvalues,
            "resid_pearson": results.resid_pearson,
            "fitted_log": np.log(results.fittedvalues)
        })
        df_resid[centered_price_col_name] = pandas_df[centered_price_col_name]
        logged_dataframes['residuals'] = df_resid

        # --- 2. Heteroscedasticity Test (Breusch-Pagan) ---
        print("   -> Running Breusch-Pagan test...")
        try:
            bp_test = het_breuschpagan(results.resid_pearson**2, results.model.exog)
            bp_p_value = bp_test[1]
            mlflow.log_metric("BP_Test_p_value", bp_p_value)
            print(f"   -> BP Test p-value: {bp_p_value:.4f}")
        except Exception as e:
            print(f"   -> WARNING: BP Test failed: {e}")

        # --- 3. Diagnostic Plots ---
        print("   -> Generating standard diagnostic plots...")

        # Histogram & Q-Q Plot
        fig_dist, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(df_resid["resid_pearson"], bins=50, kde=True, ax=axes[0])
        axes[0].set_title("Pearson Residuals Distribution")
        probplot(df_resid["resid_pearson"], dist="norm", plot=axes[1])
        plt.tight_layout()
        mlflow.log_figure(fig_dist, "visuals/02_residuals_distribution.png")
        plt.close(fig_dist)
        gc.collect()

        # Residuals vs Fitted
        fig_rvf, ax = plt.subplots(figsize=(8, 6))
        sns.scatterplot(data=df_resid, x="prediction", y="resid_pearson", alpha=0.5, s=10, ax=ax)
        ax.axhline(0, color='red', linestyle='--')
        ax.set_title("Residuals vs Fitted Values")
        mlflow.log_figure(fig_rvf, "visuals/03_residuals_vs_fitted.png")
        plt.close(fig_rvf)
        gc.collect()

        # Residuals vs Price
        fig_rvp, ax = plt.subplots(figsize=(12, 7))
        sns.scatterplot(data=df_resid, x=centered_price_col_name, y='resid_pearson', ax=ax, alpha=0.2)
        ax.axhline(0, color='red', linestyle='--')
        ax.set_title(f"Residuals vs {centered_price_col_name}")
        mlflow.log_figure(fig_rvp, f"visuals/05_residuals_vs_price.png")
        plt.close(fig_rvp)
        gc.collect()

        # --- 4. Variance Function Check (Lowess) ---
        print("   -> Generating Variance Function Check (Lowess)...")
        df_resid['resid_sq'] = df_resid['resid_pearson']**2
        
        # Sampling for plot performance
        sample_size = min(10000, len(df_resid))
        df_plot = df_resid.sample(n=sample_size, random_state=42).copy()

        fig_var, ax = plt.subplots(figsize=(12, 7))
        sns.scatterplot(data=df_plot, x='fitted_log', y='resid_sq', alpha=0.2, ax=ax)
        
        # Lowess Trend
        df_sorted = df_plot.sort_values('fitted_log')
        smoothed = lowess(df_sorted['resid_sq'], df_sorted['fitted_log'], frac=0.3)
        ax.plot(smoothed[:, 0], smoothed[:, 1], color='red', lw=2, label='Lowess Trend')
        
        ax.set_title('Variance Function Check')
        ax.set_xlabel('Log(Fitted Values)')
        ax.set_ylabel('Squared Pearson Residuals')
        mlflow.log_figure(fig_var, "visuals/06_variance_check_plot.png")
        plt.close(fig_var)
        
        del df_plot, df_sorted, smoothed; gc.collect()


        # =========================================================================================
        # STEP 9: PARTIAL DEPENDENCE PLOT (PDP)
        # =========================================================================================
        
        def _generate_and_log_pdp(df_base, price_col, model_results, log_dict):
            """
            Internal helper to calculate and log PDP.
            """
            min_val = df_base[price_col].min()
            max_val = df_base[price_col].max()
            price_grid = np.linspace(min_val, max_val, 50)
            pdp_preds = []

            # Use a working copy to avoid modifying original DF in loop
            df_work = df_base.copy()

            for p_val in price_grid:
                df_work[price_col] = p_val
                pdp_preds.append(model_results.predict(df_work).mean())
            
            del df_work

            pdp_df = pd.DataFrame({'price': price_grid, 'avg_pred': pdp_preds})
            pdp_df['log_avg_pred'] = np.log(pdp_df['avg_pred'])
            
            # Plot
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(pdp_df['price'], pdp_df['log_avg_pred'], marker='o')
            ax.set_title(f"PDP for {price_col}")
            ax.set_xlabel("Centered Price")
            ax.set_ylabel("Log Expected Sales")
            
            mlflow.log_figure(fig, f"visuals/07_pdp_{price_col}.png")
            plt.close(fig)
            
            # Save data
            log_dict[f'pdp_{price_col}'] = pdp_df

        print(f"🎨 9/12: Generating PDP for '{centered_price_col_name}'...")
        try:
            pdp_sample_n = 2000
            if len(pandas_df) > pdp_sample_n:
                df_pdp = pandas_df.sample(n=pdp_sample_n, random_state=42)
            else:
                df_pdp = pandas_df
            
            _generate_and_log_pdp(df_pdp, centered_price_col_name, results, logged_dataframes)
            
            if len(pandas_df) > pdp_sample_n: del df_pdp
            gc.collect()

        except Exception as e:
            print(f"      -> WARNING: PDP Generation failed: {e}")


        # =========================================================================================
        # STEP 10: ELASTICITY DISTRIBUTION ANALYSIS (INTERACTIONS)
        # =========================================================================================
        print("💡 10/12: Analyzing Elasticity Distribution (Interactions)...")

        try:
            interaction_prefix = f'{centered_price_col_name}:'
            df_interactions = df_coef[df_coef['term'].str.startswith(interaction_prefix)].copy()

            if df_interactions.empty:
                print("     -> WARNING: No price interactions found.")
            else:
                # Calculate Point Elasticity
                df_interactions['point_elasticity'] = base_elasticity + df_interactions['estimate']

                # Map back to context variable
                def get_context(term, vars_list):
                    clean = term.replace(interaction_prefix, "")
                    for v in vars_list:
                        if clean.startswith(v): return v
                    return 'Other'
                
                context_vars = final_categorical_vars + final_seasonal_vars
                df_interactions['context_var'] = df_interactions['term'].apply(lambda t: get_context(t, context_vars))
                
                logged_dataframes['point_elasticities'] = df_interactions

                # Boxplot
                median_order = df_interactions.groupby('context_var')['point_elasticity'].median().sort_values(ascending=False).index
                fig_h = max(6, len(median_order) * 0.6)
                
                fig_elas, ax = plt.subplots(figsize=(12, fig_h))
                sns.boxplot(x='point_elasticity', y='context_var', data=df_interactions, order=median_order, palette='coolwarm_r', ax=ax)
                
                ax.axvline(base_elasticity, color='blue', linestyle='--', label='Base Elasticity')
                ax.axvline(-1, color='gray', linestyle='-.', label='Unitary Elasticity')
                ax.set_title('Price Elasticity Distribution by Context')
                ax.legend()
                
                mlflow.log_figure(fig_elas, "visuals/12_elasticity_distribution.png")
                plt.close(fig_elas)
                gc.collect()

        except Exception as e:
            print(f"     -> ERROR in Elasticity Analysis: {e}")


        # =========================================================================================
        # STEP 11: MULTICOLLINEARITY (VIF) - OPTIMIZED WITH SAMPLING
        # =========================================================================================
        print("📊 11/12: Calculating VIF (Variance Inflation Factor)...")

        try:
            n_total = results.model.exog.shape[0]
            n_sample_vif = 5000 

            if n_total > n_sample_vif:
                print(f"    -> Sampling {n_sample_vif} rows for VIF...")
                indices = np.random.choice(n_total, n_sample_vif, replace=False)
                X_vif = results.model.exog[indices]
            else:
                X_vif = results.model.exog

            vif_data = []
            # Start range at 1 to skip Intercept
            for i in range(1, results.model.exog.shape[1]):
                try:
                    val = variance_inflation_factor(X_vif, i)
                    vif_data.append(val)
                except:
                    vif_data.append(np.nan)
            
            del X_vif; gc.collect()

            vif_df = pd.DataFrame({'feature': results.model.exog_names[1:], 'VIF': vif_data})
            logged_dataframes['vif_summary'] = vif_df

            # Log metrics
            valid_vifs = vif_df['VIF'].dropna()
            if not valid_vifs.empty:
                mlflow.log_metric("VIF_max", valid_vifs.max())
                mlflow.log_metric("VIF_mean", valid_vifs.mean())

            # VIF Boxplot
            vif_df['group'] = vif_df['feature'].apply(map_term_to_group)
            median_order = vif_df.groupby('group')['VIF'].median().sort_values(ascending=False).index
            
            fig_vif, ax = plt.subplots(figsize=(12, max(6, len(median_order)*0.5)))
            sns.boxplot(x='VIF', y='group', data=vif_df, order=median_order, palette='viridis', ax=ax)
            ax.set_xscale('log') # Log scale is crucial for VIF
            ax.axvline(10, color='red', linestyle='--', label='Critical (10)')
            ax.set_title('VIF Distribution by Group')
            
            mlflow.log_figure(fig_vif, "visuals/13_vif_distribution.png")
            plt.close(fig_vif)
            gc.collect()

        except Exception as e:
            print(f"    -> CRITICAL ERROR in VIF: {e}")
            if 'X_vif' in locals(): del X_vif
            gc.collect()


        # =========================================================================================
        # STEP 12: INTERACTION SIGNIFICANCE ANALYSIS
        # =========================================================================================
        print("💡 12/12: Analyzing Significance of Interaction Effects...")

        try:
            interaction_prefix = f'{centered_price_col_name}:'
            df_int_sig = df_coef[df_coef['term'].str.startswith(interaction_prefix)].copy()

            if not df_int_sig.empty:
                df_int_sig['group'] = df_int_sig['term'].apply(map_term_to_group)
                
                # Significance % per group
                sig_summary = df_int_sig.groupby('group')['significant'].mean() * 100
                overall_sig = df_int_sig['significant'].mean() * 100
                
                mlflow.log_metric("pct_significant_price_interactions", overall_sig)

                # Plot
                median_order = df_int_sig.groupby('group')['estimate'].median().sort_values().index
                fig_sig, ax = plt.subplots(figsize=(12, 8))
                sns.boxplot(data=df_int_sig, y='group', x='estimate', order=median_order, palette='viridis', ax=ax)
                
                # Annotate with significance %
                for i, grp in enumerate(median_order):
                    pct = sig_summary.get(grp, 0)
                    ax.text(ax.get_xlim()[1], i, f" {pct:.1f}% Sig.", va='center', fontweight='bold')

                ax.set_title('Distribution of Elasticity Modifiers (Interaction Coeffs)')
                ax.axvline(0, color='red', linestyle='--')
                
                mlflow.log_figure(fig_sig, "visuals/14_interaction_significance.png")
                plt.close(fig_sig)
                gc.collect()

        except Exception as e:
            print(f"     -> ERROR in Significance Analysis: {e}")

        # --- Final Summary Plot ---
        summary_text = results.summary().as_text()
        fig_sum, ax = plt.subplots(figsize=(14, 12))
        ax.axis('off')
        ax.text(0, 1, summary_text, fontsize=9, family='monospace', va='top')
        mlflow.log_figure(fig_sum, "visuals/01_model_summary.png")
        plt.close(fig_sum)

        # Log Summary HTML
        with open("model_summary.html", "w") as f:
            f.write(results.summary().as_html())
        mlflow.log_artifact("model_summary.html", artifact_path="diagnostics")

        display(results.summary())
        print("\n🎉 MLflow Run Completed Successfully.")
        mlflow.end_run()
        gc.collect()

    return results, df_coef, logged_dataframes


# Run Training
best_hyperparams = {'alpha': 0.5, 'L1_wt': 0.5} 
model_results, df_coefficients, dfs_debug = train_glm_model(regularize=True, best_params=best_hyperparams)


## 🧮 4. Elasticity Calculation & Shrinkage (Credibility Theory)

We apply the **Bühlmann-Straub Credibility** logic:
$$ Z = \frac{n}{n + K} $$
$$ E_{final} = Z \cdot E_{local} + (1-Z) \cdot E_{group} $$

In [ ]:
def calculate_and_shrink_elasticities(
    df: DataFrame,
    results: GLMResults,
    df_coef: pd.DataFrame,
    categorical_vars: list,
    seasonal_vars: list,
    centered_price_col: str,
    grouping_cols: list,
    # --- Robustness Parameters ---
    p_value_threshold: float = 0.10,      # Individual Filter (Statistical)
    min_individual_effect: float = 0.01,  # Individual Filter (Practical)
    min_group_avg_intensity: float = 0.02 # Group Average Intensity Filter) -> DataFrame:

    """
    Applies Bühlmann-Straub Credibility (Shrinkage) with a double-layer filtering process:
    1. Filter by Group Average Intensity (Approach 2).
    2. Individual Filter by Significance and Effect Size.
    """
    print("🚀 13/13: Calculating elasticities with Shrinkage and Intensity Filtering...")

    # Working copy of coefficients
    df_coef_clean = df_coef.copy()
    
    # ==============================================================================
    # 1. GROUP FILTER (APPROACH 2: AVERAGE INTENSITY)
    # ==============================================================================
    # "If the entire variable (e.g., TIER_BRAND) doesn't move the needle on average, discard it."
    
    vars_dropped_by_group = []
    
    print("    -> Analyzing Group Average Intensity...")
    
    # Combine categoricals and seasonal variables for verification
    all_vars_to_check = categorical_vars + seasonal_vars
    
    for var in all_vars_to_check:
        # Filter coefficients belonging to this price interaction group
        # Regex looks for: "ln_price_c:VARIABLE" or "ln_price_c:VARIABLE[...]"
        # The prefix is the interaction with price
        interaction_prefix = f'{centered_price_col}:{var}'
        
        # Select rows starting with this prefix
        mask_group = df_coef_clean['term'].str.startswith(interaction_prefix)
        
        if mask_group.any():
            # --- INTENSITY CALCULATION ---
            # Mean of absolute values of the group's coefficients
            avg_intensity = df_coef_clean.loc[mask_group, 'estimate'].abs().mean()
            
            # Decision
            if avg_intensity < min_group_avg_intensity:
                # Zero out the ENTIRE group
                df_coef_clean.loc[mask_group, 'estimate'] = 0.0
                vars_dropped_by_group.append(f"{var} (Avg: {avg_intensity:.4f})")
            else:
                # Log those that passed (optional)
                pass 

    if vars_dropped_by_group:
        print(f"    -> ✂️ Groups removed due to low average intensity (< {min_group_avg_intensity}):")
        for v in vars_dropped_by_group:
            print(f"         - {v}")
    else:
        print("    -> No entire groups were removed (all meet minimum average relevance).")


    # ==============================================================================
    # 2. INDIVIDUAL FILTER (SIGNIFICANCE AND POINT MAGNITUDE)
    # ==============================================================================
    # Now look coefficient by coefficient within the remaining groups.
    
    is_interaction = df_coef_clean['term'].str.contains(':')
    
    # Individual criteria
    cond_stat = df_coef_clean['p_value'] < p_value_threshold
    cond_practical = df_coef_clean['estimate'].abs() > min_individual_effect
    
    # If already zeroed in step 1, it remains zero. If not, apply individual filter.
    # If estimate is 0.0 (from step 1), it fails cond_practical, staying 0.0.
    mask_noise = is_interaction & (~cond_stat | ~cond_practical)
    
    # Count only 'individually zeroed' if not zeroed before (for accurate printing)
    n_zeroed_individual = mask_noise.sum()
    
    # Zero out
    df_coef_clean.loc[mask_noise, 'estimate'] = 0.0
    
    print(f"    -> 🧹 Fine Tuning: {n_zeroed_individual} specific coefficients zeroed due to p-value or irrelevant size.")

    # ==============================================================================
    # 3. DICTIONARY CONSTRUCTION AND SPARK MAPPING
    # ==============================================================================
    # Extract Base Elasticity
    base_elasticity = float(df_coef_clean.loc[df_coef_clean['term'] == centered_price_col, 'estimate'].iloc[0])
    
    # Extract Dispersion (Scale) from Results
    dispersion = float(results.scale)
    
    # Convert Coef DataFrame to Dictionary for fast lookup
    coef_dict = df_coef_clean.set_index('term')['estimate'].astype(float).to_dict()

    # Create Mapping DataFrames for Categorical Variables
    mapping_dfs = {}
    for var in categorical_vars:
        interaction_prefix = f'{centered_price_col}:{var}'
        # Get coefficients (now cleaned/zeroed)
        interaction_terms = {k: v for k, v in coef_dict.items() if k.startswith(interaction_prefix)}

        entries = []
        for term, estimate in interaction_terms.items():
            # Regex to extract level name: "ln_price:BRAND[T.BrandA]" -> "BrandA"
            match = re.search(r'\[T\.(.*?)\]', term)
            if match:
                entries.append((match.group(1), float(estimate)))

        if entries:
            # Create small DF to join back
            schema = StructType([
                StructField(var, StringType(), False),
                StructField(f"{var}_coef", DoubleType(), False)])
            mapping_dfs[var] = df.sparkSession.createDataFrame(entries, schema=schema)
        else:
            mapping_dfs[var] = None

    # Apply Mappings to Main Spark DataFrame
    df_local = df
    
    # Map Seasonal Variables (Scalars)
    for s in seasonal_vars:
        coef_s = coef_dict.get(f'{centered_price_col}:{s}', 0.0)
        df_local = df_local.withColumn(f"{s}_coef", F.lit(float(coef_s)))
        
    # Map Categorical Variables (Joins)
    for v, map_df in mapping_dfs.items():
        if map_df is not None:
            # Join and fill nulls with 0 (reference category or dropped coeff)
            df_local = df_local.join(map_df, on=v, how='left').fillna(0.0, subset=[f"{v}_coef"])
        else:
            df_local = df_local.withColumn(f"{v}_coef", F.lit(0.0))

    # Calculate Local Elasticity
    # E_local = Base + Sum(Cat_Coefs) + Sum(Seasonal * Seasonal_Coefs)
    expr = F.lit(base_elasticity)
    for v in categorical_vars:
        expr = expr + F.col(f"{v}_coef")
    for s in seasonal_vars:
        expr = expr + F.col(s) * F.col(f"{s}_coef")
        
    df_local = df_local.withColumn('elasticity_local', expr)

    # ==============================================================================
    # 4. SHRINKAGE CALCULATION (BÜHLMANN-STRAUB)
    # ==============================================================================
    
    # Calculate Group Stats (Prior)
    df_group_mean = df_local.groupBy(*grouping_cols).agg(F.mean('elasticity_local').alias('elasticity_group_mean'))
    
    # Calculate Variance Between Groups (Tau2)
    tau2_val = df_group_mean.select(F.var_samp('elasticity_group_mean').alias('tau2')).first()['tau2']
    tau2 = tau2_val if tau2_val is not None else 0.0
    
    # Calculate Kappa (Credibility Constant)
    if tau2 < 1e-6:
        kappa = float('inf')
        print("    -> WARNING: Variance between groups is near zero (tau2 ~ 0). Applying Full Shrinkage to global mean.")
    else:
        kappa = dispersion / tau2
        print(f"    -> Shrinkage Stats: Dispersion={dispersion:.4f}, Tau2={tau2:.5f}, Kappa={kappa:.2f}")

    # Define Windows
    context_cols = [c for c in categorical_vars if c not in grouping_cols]
    
    if context_cols:
        win_ctx = Window.partitionBy(*[F.col(c) for c in context_cols])
    else:
        win_ctx = Window.partitionBy()
    
    # Count observations per context (n_i)
    df_local = df_local.withColumn('n_i', F.count('*').over(win_ctx))
    
    # Get Group Mean on main DF
    win_group = Window.partitionBy(*[F.col(c) for c in grouping_cols])
    df_local = df_local.withColumn('elasticity_group_mean', F.mean('elasticity_local').over(win_group))

    # Calculate Shrinkage Weight (Z) and Final Elasticity
    # Z = n / (n + K)
    df_local = df_local.withColumn(
        'shrinkage_weight',
        F.col('n_i') / (F.col('n_i') + F.lit(kappa))).withColumn(
        'elasticity_final',
        (F.col('shrinkage_weight') * F.col('elasticity_local')) +
        ((1 - F.col('shrinkage_weight')) * F.col('elasticity_group_mean')))

    # ==============================================================================
    # 5. FINAL ADJUSTMENTS (CAPPING AND POSITIVE HANDLING)
    # ==============================================================================
    
    # Cap at Group Median to prevent outliers
    win_group_median = Window.partitionBy(*grouping_cols)
    df_local = df_local.withColumn(
        'group_median',
        F.percentile_approx('elasticity_final', 0.5).over(win_group_median))
    
    # Logic: If final > median, cap it (assuming we want more negative/elastic values or conservative estimates)
    df_local = df_local.withColumn(
        'elasticity_final',
        F.when(F.col('elasticity_final') > F.col('group_median'), F.col('group_median'))
          .otherwise(F.col('elasticity_final')))

    # Handle Positive Elasticities (Giffen Goods Check)
    # Calculate Global Median as fallback
    mediana_global_val = df_local.agg(F.expr('percentile_approx(elasticity_final, 0.5)').alias('mediana_global')).first()['mediana_global']
    
    # Force negative fallback if even the global median is positive (rare)
    if mediana_global_val > 0: 
        mediana_global_val = -0.1

    # Apply fallback: If elasticity > 0, replace with global median
    df_local = df_local.withColumn(
        'elasticity_final',
        F.when(F.col('elasticity_final') > 0, F.lit(mediana_global_val))
          .otherwise(F.col('elasticity_final')))

    print("✅ Elasticities calculated, filtered, and regularized.")
    return df_local

In [ ]:
# Execute Calculation
grouping_cols_shrinkage = ['TIER_BRAND'] # Shrink towards Price Tier

df_final_elasticity = calculate_and_shrink_elasticities(
    df=df_feat,
    results=model_results,
    df_coef=df_coefficients,
    categorical_vars=final_categorical_vars,
    seasonal_vars=final_seasonal_vars,
    centered_price_col=centered_price_col_name,
    grouping_cols=grouping_cols_shrinkage,
    p_value_threshold=0.10,
    min_individual_effect=0.01,
    min_group_avg_intensity=0.02)

In [ ]:
# Comparison Histogram (Pandas)
pdf_viz = df_final_elasticity.select("elasticity_final", "elasticity_local").sample(0.01).toPandas()

plt.figure(figsize=(10, 6))
sns.histplot(data=pdf_viz, x="elasticity_final", color="skyblue", label="Final (Shrunk)", kde=True)
sns.histplot(data=pdf_viz, x="elasticity_local", color="salmon", label="Local (Raw)", kde=True)
plt.title('Distribution of Elasticities: Local vs Shrunk')
plt.legend()
display(plt.gcf())


## 💾 5. Saving Trusted Data

In [ ]:
target_table = "pricing_db.gold_elasticity_model_trusted"

df_final_elasticity.drop("features").write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

spark.sql(f"""
  COMMENT ON TABLE {target_table}
  IS 'Trusted elasticity table containing Local and Shrunk elasticities per SKU/Context.'
""")

print("✅ Pipeline Completed Successfully.")